# TF-IDF Baseline
### NLP fundamentals: tokenization, stop words, TF-IDF

Builds a TF-IDF baseline that matches resumes to job descriptions:
1. Tokenize + remove stop words (NLTK)
2. Vectorize resumes and job descriptions with TF-IDF (shared vocabulary)
3. Score every resume against every job description with cosine similarity
4. For a chosen job description, output the top-ranked resumes

This is the baseline the Sentence-Transformers embeddings stage (next) will be compared against.

Assumes `preprocess.ipynb` has already been run (so `data/processed/resumes_clean.csv`
and `data/processed/job_descriptions.csv` exist).


In [ ]:
%run config.ipynb

In [10]:
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 2.4 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.6 MB/s  0:00:00
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)

   ---------------------------------------- 0/3 [regex]
   ---------------------------------------- 0/3 [regex]
   ------------- -------------------------- 1/3 [defusedxml]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\RanVic\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Load cleaned data

In [ ]:
# Paths come from config.ipynb (PROCESSED_RESUMES_CSV, JOB_DESCRIPTIONS_CSV), so this
# notebook runs on any machine. (Sprint 4: the hard-coded Windows paths that the new CI
# notebook check flagged were removed from this cell.)
print(PROCESSED_RESUMES_CSV)
print(JOB_DESCRIPTIONS_CSV)

In [8]:
import pandas as pd

resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
job_descriptions = pd.read_csv(JOB_DESCRIPTIONS_CSV)

print(f"Loaded {len(resumes_clean)} resumes and {len(job_descriptions)} job descriptions")
resumes_clean.head()

Loaded 2481 resumes and 24 job descriptions


,ID,Category,clean_text
0,16852973,HR,HR ADMINISTRATOR/MARKETING ASSOCIATE HR ADMINI...
1,22323967,HR,"HR SPECIALIST, US HR OPERATIONS Summary Versat..."
2,33176873,HR,HR DIRECTOR Summary Over 20 years experience i...
3,27018550,HR,"HR SPECIALIST Summary Dedicated, Driven, and D..."
4,17812897,HR,HR MANAGER Skill Highlights HR SKILLS HR Depar...


## 2. Tokenization & stop words
Uses NLTK's stopword list (more complete than the small built-in one used during
preprocessing) plus simple regex tokenization — lowercase, strip punctuation/numbers,
keep words of length >= 2.

In [11]:
import re
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))

def tokenize(text: str) -> list[str]:
    """Lowercase, strip non-letters, split on whitespace, drop stopwords/short tokens."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

# Quick sanity check
sample_tokens = tokenize(resumes_clean.loc[0, "clean_text"])
print(sample_tokens[:20])

['administrator', 'marketing', 'associate', 'administrator', 'summary', 'dedicated', 'customer', 'service', 'manager', 'years', 'experience', 'hospitality', 'customer', 'service', 'management', 'respected', 'builder', 'leader', 'customer', 'focused']


## 3. TF-IDF vectorization
Fit one TF-IDF vectorizer over the combined vocabulary of resumes + job descriptions,
so both sides live in the same vector space and cosine similarity is meaningful.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tokenizer_joiner(text: str) -> str:
    """TfidfVectorizer expects a callable that returns a string or token list;
    we reuse our own tokenize() so stopword handling stays consistent."""
    return tokenize(text)

vectorizer = TfidfVectorizer(tokenizer=tokenizer_joiner, lowercase=False, token_pattern=None)

# Fit on the combined corpus so resumes and job descriptions share one vocabulary
combined_corpus = pd.concat([
    resumes_clean["clean_text"],
    job_descriptions["job_description"],
], ignore_index=True)

vectorizer.fit(combined_corpus)

resume_vectors = vectorizer.transform(resumes_clean["clean_text"])
job_vectors = vectorizer.transform(job_descriptions["job_description"])

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Resume matrix shape:", resume_vectors.shape)
print("Job description matrix shape:", job_vectors.shape)

Vocabulary size: 37136
Resume matrix shape: (2481, 37136)
Job description matrix shape: (24, 37136)


## 4. Score resumes against a job description

In [14]:
COL_ID = "ID"
COL_CATEGORY = "Category"

from sklearn.metrics.pairwise import cosine_similarity

def top_matches_for_category(category: str, top_k: int = 5) -> pd.DataFrame:
    """Return the top_k resumes ranked by TF-IDF cosine similarity to the given
    job description category."""
    job_idx = job_descriptions.index[job_descriptions[COL_CATEGORY] == category]
    if len(job_idx) == 0:
        raise ValueError(f"No job description found for category: {category}")
    job_idx = job_idx[0]

    sims = cosine_similarity(job_vectors[job_idx], resume_vectors).flatten()
    results = resumes_clean.copy()
    results["similarity"] = sims
    return results.sort_values("similarity", ascending=False).head(top_k)[
        [COL_ID, COL_CATEGORY, "similarity"]
    ]

# Example: try any category present in your data, e.g. the first one available
example_category = job_descriptions[COL_CATEGORY].iloc[0]
print(f"Top matches for category: {example_category}\n")
top_matches_for_category(example_category)

Top matches for category: ACCOUNTANT



,ID,Category,similarity
1899,25867805,ACCOUNTANT,0.548370
1806,15906625,ACCOUNTANT,0.531392
1484,78229715,FINANCE,0.530576
1866,49204385,ACCOUNTANT,0.488704
1828,21338490,ACCOUNTANT,0.484945


## 5. Save the vectorizer + matrices (optional, for reuse in later stages)

In [16]:
import joblib

TFIDF_DIR = PROCESSED_DATA_DIR / "tfidf"
TFIDF_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, TFIDF_DIR / "tfidf_vectorizer.joblib")
joblib.dump(resume_vectors, TFIDF_DIR / "resume_vectors.joblib")
joblib.dump(job_vectors, TFIDF_DIR / "job_vectors.joblib")

print(f"Saved TF-IDF artifacts to {TFIDF_DIR}")

---
## Notes
- This is a **baseline**: TF-IDF only captures exact word overlap (after stopword removal),
  so it will miss semantic matches (e.g. "developer" vs "engineer"). That gap is exactly
  what the next stage — Sentence-Transformers embeddings — is meant to close.
- The vectorizer is fit on the combined resume + job description corpus so both live in
  the same vector space; refitting on new data later will change the vocabulary, so the
  saved `.joblib` files should be treated as tied to this dataset snapshot.
